# AST Scream Detection
Binary scream / non-scream classification using Audio Spectrogram Transformer (AST).
Fine-tuned on two public Kaggle datasets.

In [ ]:
import os, random, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_score, recall_score, f1_score,
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import ASTFeatureExtractor, ASTForAudioClassification

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

## Configuration
- **Dataset 1** Audio Dataset of Scream and Non-Scream : 1,583 scream / 1,545 non-scream (nearly balanced), 44,100 Hz
- **Dataset 2** Human Screaming Detection Dataset : 862 scream / 2,631 non-scream  (imbalanced) , 44,100 Hz
- Resampled to **16,000 Hz** to match AST pre-training on AudioSet

In [ ]:
DATASET_1     = '/kaggle/input/datasets/aananehsansiam/audio-dataset-of-scream-and-non-scream'
DS1_SCREAM    = os.path.join(DATASET_1, 'Converted_Separately', 'scream')
DS1_NONSCREAM = os.path.join(DATASET_1, 'Converted_Separately', 'non_scream')

DATASET_2     = '/kaggle/input/datasets/whats2000/human-screaming-detection-dataset'
DS2_SCREAM    = os.path.join(DATASET_2, 'Screaming')
DS2_NONSCREAM = os.path.join(DATASET_2, 'NotScreaming')

TARGET_SR     = 16_000          # AST pre-trained at 16 kHz
CLIP_DURATION = 4.0             # seconds
TARGET_LENGTH = int(TARGET_SR * CLIP_DURATION)  # 64,000 samples
SEED          = 42
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Device: {DEVICE} | Target SR: {TARGET_SR} Hz | Clip: {TARGET_LENGTH} samples')

## File Collection and Balancing

In [ ]:
def collect_files(folder, label, source_name):
    rows, folder = [], Path(folder)
    if not folder.exists():
        print(f'[WARNING] Missing: {folder}')
        return rows
    for fp in folder.rglob('*.wav'):
        rows.append({'filepath': str(fp), 'label': label, 'source': source_name})
    return rows

rows  = collect_files(DS1_SCREAM,    1, 'dataset1')
rows += collect_files(DS1_NONSCREAM, 0, 'dataset1')
rows += collect_files(DS2_SCREAM,    1, 'dataset2')
rows += collect_files(DS2_NONSCREAM, 0, 'dataset2')
df = pd.DataFrame(rows)
print(df.groupby(['source', 'label']).size().rename('n').reset_index().to_string(index=False))

In [ ]:
def balance_binary_source(df_source, seed=42):
    """Undersample majority class within one source until pos == neg."""
    pos = df_source[df_source['label'] == 1]
    neg = df_source[df_source['label'] == 0]
    n   = min(len(pos), len(neg))
    return pd.concat([
        pos.sample(n=n, random_state=seed),
        neg.sample(n=n, random_state=seed),
    ]).sample(frac=1, random_state=seed).reset_index(drop=True)

def stratified_sample(df_bal, n_total, seed=42):
    n   = n_total // 2
    pos = df_bal[df_bal['label'] == 1].sample(n=min(n, (df_bal['label']==1).sum()), random_state=seed)
    neg = df_bal[df_bal['label'] == 0].sample(n=min(n, (df_bal['label']==0).sum()), random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)

df1_bal  = balance_binary_source(df[df['source'] == 'dataset1'], seed=SEED)
df2_bal  = balance_binary_source(df[df['source'] == 'dataset2'], seed=SEED)
n_source = min(len(df1_bal), len(df2_bal))
df1_bal  = stratified_sample(df1_bal, n_source, seed=SEED)
df2_bal  = stratified_sample(df2_bal, n_source, seed=SEED)
df       = pd.concat([df1_bal, df2_bal]).sample(frac=1, random_state=SEED).reset_index(drop=True)

counts = df['label'].value_counts()
assert counts[0] == counts[1], f'Imbalanced: {counts.to_dict()}'
print(f'Balanced dataset: {len(df)} samples')
print(df.groupby(['source', 'label']).size().rename('n').reset_index().to_string(index=False))

In [ ]:
def is_valid_audio(fp):
    try:
        librosa.load(fp, sr=None, mono=False, duration=0.1)
        return True
    except Exception:
        return False

valid_mask = df['filepath'].apply(is_valid_audio)
df = df[valid_mask].reset_index(drop=True)
print(f'Valid files: {len(df)}')

## Train / Val / Test Split — 80 / 10 / 10
Stratified by `source × label` to preserve all four group proportions across splits.

In [ ]:
df['stratify_key'] = df['source'] + '_' + df['label'].astype(str)

train_df, temp_df = train_test_split(df,      test_size=0.20, stratify=df['stratify_key'],      random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['stratify_key'], random_state=SEED)

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    d = split['label'].value_counts(normalize=True)
    print(f'{name:5s} ({len(split):4d})  scream={d.get(1,0):.3f}  non-scream={d.get(0,0):.3f}')

## Audio Loading and Preprocessing

In [ ]:
def load_audio(filepath):
    y, sr = librosa.load(filepath, sr=None, mono=False)
    if y.ndim > 1:
        y = np.mean(y, axis=0)                            
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR, res_type='soxr_hq')  
    return y.astype(np.float32)

def crop_or_pad(y, target_length=TARGET_LENGTH, is_train=False):
    if len(y) < target_length:
        return np.pad(y, (0, target_length - len(y)), mode='constant')
    if len(y) == target_length:
        return y
    start = np.random.randint(0, len(y) - target_length + 1) if is_train             else (len(y) - target_length) // 2
    return y[start: start + target_length]

def energy_based_crop(y, target_length=TARGET_LENGTH, is_train=False):
    """Centre crop on peak RMS energy frame; add random jitter during training."""
    if len(y) <= target_length:
        return np.pad(y, (0, max(0, target_length - len(y))), mode='constant')
    energy = librosa.feature.rms(y=y, frame_length=512, hop_length=256).squeeze()
    peak   = int(np.argmax(energy)) * 256
    jitter = int(np.random.randint(-TARGET_SR // 2, TARGET_SR // 2)) if is_train else 0
    center = int(np.clip(peak + jitter, 0, len(y) - 1))
    start  = max(0, center - target_length // 2)
    end    = start + target_length
    if end > len(y):
        end, start = len(y), len(y) - target_length
    return y[start:end]

print('Audio utilities defined.')

## Waveform Augmentation (training only)


In [ ]:
def time_shift(y, shift_max=0.2):
    
    max_shift = int(len(y) * shift_max)
    shift     = np.random.randint(-max_shift, max_shift + 1)
    out       = np.zeros_like(y)
    if   shift > 0: out[shift:]  = y[:-shift]
    elif shift < 0: out[:shift]  = y[-shift:]
    else:           out[:]       = y
    return out

def add_noise(y, noise_factor=0.003):
    return np.clip(y + noise_factor * np.random.randn(len(y)).astype(np.float32), -1.0, 1.0)

def augment_audio(y):
    if np.random.rand() < 0.5:
        y = time_shift(y, shift_max=0.2)                                          
    if np.random.rand() < 0.5:
        y = add_noise(y, noise_factor=float(np.random.uniform(0.002, 0.01)))
    if np.random.rand() < 0.5:
        y = np.clip(y * float(np.random.uniform(0.7, 1.3)), -1.0, 1.0)
    if np.random.rand() < 0.3:
        y = librosa.effects.pitch_shift(y, sr=TARGET_SR, n_steps=float(np.random.uniform(-1.0, 1.0)))
    if np.random.rand() < 0.3:
        y = librosa.effects.time_stretch(y, rate=float(np.random.uniform(0.9, 1.1)))
        y = crop_or_pad(y, target_length=TARGET_LENGTH, is_train=True)            # FIX-2
    return y

print('Waveform augmentation defined.')

## Spectrogram Augmentation SpecAugment (training only)

In [ ]:
def spec_augment_ast(spec, freq_mask_param=16, time_mask_param=64):
    spec = spec.clone()
    if torch.rand(1).item() < 0.5:                                # frequency masking (axis 1)
        f = torch.randint(0, freq_mask_param + 1, (1,)).item()
        if f > 0 and spec.shape[1] > f:
            f0 = torch.randint(0, spec.shape[1] - f, (1,)).item()
            spec[:, f0: f0 + f] = 0.0
    if torch.rand(1).item() < 0.5:                                # time masking (axis 0)
        t = torch.randint(0, time_mask_param + 1, (1,)).item()
        if t > 0 and spec.shape[0] > t:
            t0 = torch.randint(0, spec.shape[0] - t, (1,)).item()
            spec[t0: t0 + t, :] = 0.0
    return spec

print('SpecAugment defined.')

## Dataset and DataLoaders

In [ ]:
feature_extractor = ASTFeatureExtractor(
    sampling_rate=TARGET_SR,
    num_mel_bins=128,
    max_length=1024,
)

class ASTScreamDataset(Dataset):
    def __init__(self, dataframe, is_train=False):
        self.df       = dataframe.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y   = load_audio(row['filepath'])
        y   = energy_based_crop(y, target_length=TARGET_LENGTH, is_train=self.is_train)
        if self.is_train:
            y = augment_audio(y)
        inputs       = feature_extractor(y, sampling_rate=TARGET_SR, return_tensors='pt')
        input_values = inputs['input_values'].squeeze(0)
        if self.is_train:
            input_values = spec_augment_ast(input_values, freq_mask_param=16, time_mask_param=64)
        return input_values, torch.tensor(int(row['label']), dtype=torch.long)

train_dataset = ASTScreamDataset(train_df, is_train=True)
val_dataset   = ASTScreamDataset(val_df,   is_train=False)
test_dataset  = ASTScreamDataset(test_df,  is_train=False)
print('Datasets created.')

In [ ]:
# WeightedRandomSampler: weight each sample by 1/group_size so all
# four (source x label) groups are sampled equally per batch
train_keys     = train_df['source'] + '_' + train_df['label'].astype(str)
key_counts     = train_keys.value_counts().to_dict()
sample_weights = train_keys.map(lambda x: 1.0 / key_counts[x]).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False,     num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False,     num_workers=2, pin_memory=True)
print('DataLoaders ready.')

## Model, Optimizer, and Scheduler

In [ ]:
model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=2,
    ignore_mismatched_sizes=True,   # replaces 527-class head with new 2-class head
).to(DEVICE)

criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
NUM_EPOCHS = 15
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)  

print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x).logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_labels, all_probs = 0.0, [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        logits = model(x).logits
        probs  = torch.softmax(logits, dim=1)[:, 1]
        total_loss  += criterion(logits, y).item() * x.size(0)
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    return total_loss / len(loader.dataset), all_labels, all_probs, (all_probs >= 0.5).astype(int)

print('Functions defined.')

## Training Loop

In [ ]:
PATIENCE         = 6          
best_val_f1      = -1.0
patience_counter = 0
history          = {'train_loss': [], 'val_loss': [], 'val_macro_f1': [], 'lr': []}

for epoch in range(NUM_EPOCHS):
    train_loss                                    = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_labels, val_probs, val_preds    = evaluate(model, val_loader, criterion, DEVICE)
    val_macro_f1 = f1_score(val_labels, val_preds, average='macro')
    current_lr   = scheduler.get_last_lr()[0]

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_macro_f1'].append(val_macro_f1)
    history['lr'].append(current_lr)

    saved = val_macro_f1 > best_val_f1
    print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  LR={current_lr:.2e}  Train={train_loss:.4f}  Val={val_loss:.4f}  F1={val_macro_f1:.4f}' + ('  <- saved' if saved else ''))

    if saved:
        best_val_f1      = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), 'best_ast_scream.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print('Early stopping.')
            break

    scheduler.step()   

print(f'Best val macro F1: {best_val_f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['val_macro_f1'], color='green'); axes[1].set_title('Val Macro F1'); axes[1].set_xlabel('Epoch')
axes[2].plot(history['lr'], color='orange'); axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

## Threshold Selection on Validation Set

In [ ]:
model.load_state_dict(torch.load('best_ast_scream.pth', map_location=DEVICE))
model.eval()

_, val_labels, val_probs, _ = evaluate(model, val_loader, criterion, DEVICE)

best_threshold, best_score = 0.5, -1.0
print(f"\n{'Threshold':>10} | {'Precision':>10} | {'Recall':>8} | {'F1':>8}")
print('-' * 50)
for th in np.arange(0.30, 0.71, 0.05):
    preds = (val_probs >= th).astype(int)
    prec  = precision_score(val_labels, preds, zero_division=0)
    rec   = recall_score(val_labels, preds, zero_division=0)
    f1    = f1_score(val_labels, preds, zero_division=0)
    score = f1 if prec >= 0.75 else -1.0   # precision floor prevents trivial all-scream solution
    marker = '  <- best' if score > best_score else ''
    print(f'{th:>10.2f} | {prec:>10.4f} | {rec:>8.4f} | {f1:>8.4f}{marker}')
    if score > best_score:
        best_score, best_threshold = score, th

print(f'\nChosen threshold: {best_threshold:.2f}')

## Test Set Evaluation

In [ ]:
_, test_labels, test_probs, _ = evaluate(model, test_loader, criterion, DEVICE)
test_preds = (test_probs >= best_threshold).astype(int)

print('=' * 60)
print(f'TEST SET RESULTS  |  threshold={best_threshold:.2f}')
print('=' * 60)
print(classification_report(test_labels, test_preds, target_names=['Non-scream', 'Scream'], digits=4))

cm = confusion_matrix(test_labels, test_preds)
print(pd.DataFrame(cm, index=['True Non-scream', 'True Scream'],
                       columns=['Pred Non-scream', 'Pred Scream']).to_string())
print(f'\nROC-AUC: {roc_auc_score(test_labels, test_probs):.4f}')

In [ ]:
# Per-source breakdown — verifies generalisation across both datasets
test_results = test_df.copy().reset_index(drop=True)
test_results['label'] = test_labels.astype(int)
test_results['pred']  = test_preds.astype(int)
test_results['prob']  = test_probs

rows_out = []
for src in sorted(test_results['source'].unique()):
    sub = test_results[test_results['source'] == src]
    rep = classification_report(sub['label'], sub['pred'], digits=4, output_dict=True, zero_division=0)
    key = 1 if 1 in rep else '1'
    rows_out.append({
        'source':    src, 'n': len(sub),
        'accuracy':  round(rep['accuracy'], 4),
        'macro_f1':  round(rep['macro avg']['f1-score'], 4),
        'precision': round(rep[key]['precision'], 4),
        'recall':    round(rep[key]['recall'], 4),
        'scream_f1': round(rep[key]['f1-score'], 4),
    })
print(pd.DataFrame(rows_out).to_string(index=False))

## Save Model

In [ ]:
SAVE_DIR = '/kaggle/working/scream_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.load_state_dict(torch.load('best_ast_scream.pth', map_location=DEVICE))
model.eval()

torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'model_state_dict.pth'))

with open(os.path.join(SAVE_DIR, 'config.json'), 'w') as f:
    json.dump({
        'model_name':        'MIT/ast-finetuned-audioset-10-10-0.4593',
        'num_labels':        2,
        'id2label':          {'0': 'Non-Scream', '1': 'Scream'},
        'label2id':          {'Non-Scream': 0, 'Scream': 1},
        'sampling_rate':     TARGET_SR,
        'best_threshold':    float(round(best_threshold, 4)),
        'best_val_macro_f1': float(round(best_val_f1, 4)),
        'weight_file':       'model_state_dict.pth',
        'task':              'scream_detection',
    }, f, indent=2)

with open(os.path.join(SAVE_DIR, 'label_mapping.json'), 'w') as f:
    json.dump({'0': 'Non-Scream', '1': 'Scream'}, f, indent=2)

feature_extractor.save_pretrained(SAVE_DIR)

zip_path = shutil.make_archive('/kaggle/working/scream_model', 'zip', '/kaggle/working', 'scream_model')
print('Saved:')
for fname in sorted(os.listdir(SAVE_DIR)):
    print(f'  {fname:<30} {os.path.getsize(os.path.join(SAVE_DIR, fname))/1e6:.1f} MB')
print(f'Zip: {zip_path}')
print(f'Threshold: {best_threshold:.2f}  |  Best val F1: {best_val_f1:.4f}')

In [ ]:
import librosa.display

samples = pd.concat([
    test_df[test_df['label'] == 1].sample(2, random_state=42),
    test_df[test_df['label'] == 0].sample(2, random_state=42),
]).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

for i, row in samples.iterrows():
    y    = load_audio(row['filepath'])
    mel  = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_fft=400, hop_length=160, n_mels=128, power=2.0)
    spec = librosa.power_to_db(mel, ref=np.max)
    img  = librosa.display.specshow(spec, sr=TARGET_SR, hop_length=160, x_axis='time', y_axis='mel', ax=axes[i], cmap='magma')
    axes[i].set_title(f'{["Non-Scream","Scream"][row["label"]]}  —  {row["source"]}', fontweight='bold')
    fig.colorbar(img, ax=axes[i], format='%+2.0f dB', shrink=0.85)

plt.suptitle('Mel Spectrograms — Scream vs Non-Scream', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('spectrograms.png', dpi=200, bbox_inches='tight')
plt.show()